
### 15 - Endpoint Testing

##### Purpose

This notebook validates the deployed support-ticket Transformer
through the actual Databricks Model Serving endpoint.

Unlike previous notebooks that loaded the MLflow model directly,
this notebook sends requests through the network serving layer.

The notebook verifies:

- endpoint readiness
- deployed model/version
- single-ticket inference
- batch inference
- response structure
- predicted categories
- class probabilities
- basic behavioral test cases

This notebook is independently runnable and does not depend on
execution state from any previous notebook.

##### Technologies

- Databricks Model Serving
- Databricks SDK for Python
- Unity Catalog Model Registry
- MLflow PyFunc
- pandas


##### Input

Raw support-ticket text: My account is locked and I cannot login.


##### Output

Expected serving response contains:

- predicted_category
- confidence
- probability_Billing
- probability_Cancellation
- probability_Login
- probability_Technical

##### Architecture

Endpoint Testing Flow :

``` text

Raw text
   ↓
dataframe_records
   ↓
Databricks Serving Endpoint
   ↓
MLflow PyFunc
   ↓
Fine-tuned Transformer
   ↓
Prediction response

```

The REST endpoint supports dataframe_records as one of the standard request formats for custom ML models.

##### 1. Imports

In [0]:
import pandas as pd

from databricks.sdk import WorkspaceClient

from src.project_config import (
    SERVING_ENDPOINT_NAME,
    REGISTERED_MODEL_NAME,
)

##### 2. Initialize Databricks client

In [0]:
workspace_client = WorkspaceClient()

print(
    "Serving endpoint:",
    SERVING_ENDPOINT_NAME
)

print(
    "Registered model:",
    REGISTERED_MODEL_NAME
)

##### 3. Verify endpoint readiness

Before sending any prediction request, verify that the endpoint is queryable.

Databricks defines an endpoint as READY when its active served entities are ready to receive queries.

In [0]:

endpoint = (
    workspace_client
    .serving_endpoints
    .get(
        SERVING_ENDPOINT_NAME
    )
)

if endpoint.state is None:
    raise RuntimeError(
        "Serving endpoint state is unavailable."
    )

ready_state = (
    endpoint.state.ready
)

config_update_state = (
    endpoint.state.config_update
)

print(
    "Ready:",
    ready_state
)

print(
    "Config update:",
    config_update_state
)

In [0]:
if (
    str(ready_state)
    != "EndpointStateReady.READY"
):
    raise RuntimeError(
        "Serving endpoint is not ready."
    )

if (
    str(config_update_state)
    !=
    "EndpointStateConfigUpdate.NOT_UPDATING"
):
    raise RuntimeError(
        "Serving endpoint configuration "
        "is still updating."
    )

print(
    "Endpoint is ready for inference."
)

##### 4. Verify deployed served entity

In [0]:
#This proves which model version the endpoint is currently serving.
endpoint =     workspace_client.serving_endpoints.get(SERVING_ENDPOINT_NAME)

served_entities = (
    endpoint.config.served_entities
    if endpoint.config
    else None
)

if not served_entities:
    raise RuntimeError(
        "No served entities were found."
    )

In [0]:
served_entities

In [0]:
for entity in served_entities:

    print(
        "Served entity:",
        entity.name
    )

    print(
        "Registered model:",
        entity.entity_name
    )

    print(
        "Model version:",
        entity.entity_version
    )

    print(
        "Scale to zero:",
        entity.scale_to_zero_enabled
    )

    print("-" * 60)

##### 5.Single-ticket endpoint test

In [0]:
single_ticket = {
    "ticket_text":
        "My account is locked and  I cannot login."
}

In [0]:
single_response = (
    workspace_client
    .serving_endpoints
    .query(
        name=SERVING_ENDPOINT_NAME,
        dataframe_records=[
            single_ticket
        ],
    )
)

#Databricks' SDK specifically supports dataframe_records for pandas-style row-oriented model inputs.

In [0]:
print(
    single_response
)

##### 6. Inspect the response structure

In [0]:
print(
    "Response type:",
    type(single_response)
)

In [0]:
print(
    "Predictions:",
    single_response.predictions
)

##### 7. Convert prediction to DataFrame

In [0]:
single_prediction_df = (
    pd.DataFrame(
        single_response.predictions
    )
)

display(
    single_prediction_df
)

In [0]:
expected_output_columns = {
    "predicted_category",
    "confidence",
    "probability_Billing",
    "probability_Cancellation",
    "probability_Login",
    "probability_Technical",
}

missing_columns = (
    expected_output_columns
    -
    set(
        single_prediction_df.columns
    )
)

if missing_columns:
    raise ValueError(
        "Endpoint response is missing "
        f"expected columns: {missing_columns}"
    )

print(
    "Endpoint response schema verified."
)

##### 8. Validate probability behavior

In [0]:
#Now verify that our multiclass output still behaves correctly after deployment.

probability_columns = [
    "probability_Billing",
    "probability_Cancellation",
    "probability_Login",
    "probability_Technical",
]

probability_sum = (
    single_prediction_df[
        probability_columns
    ]
    .sum(axis=1)
    .iloc[0]
)

print(
    "Probability sum:",
    probability_sum
)

In [0]:
assert abs(
    probability_sum - 1.0
) < 1e-5

print(
    "Class probabilities sum to 1."
)

In [0]:
max_probability = (
    single_prediction_df[
        probability_columns
    ]
    .max(axis=1)
    .iloc[0]
)

reported_confidence = (
    single_prediction_df[
        "confidence"
    ]
    .iloc[0]
)

assert abs(
    max_probability
    - reported_confidence
) < 1e-5

print(
    "Confidence matches the maximum class probability."
)

4 logits
   ↓
softmax
   ↓
4 class probabilities
   ↓
argmax
   ↓
predicted category

max(probabilities)
   ↓
confidence

##### 9. Batch endpoint test

In [0]:
batch_tickets = [
    {
        "ticket_text":
            "I was charged twice for my monthly bill."
    },
    {
        "ticket_text":
            "Please cancel my subscription."
    },
    {
        "ticket_text":
            "My account is locked and I cannot login."
    },
    {
        "ticket_text":
            "My internet connection keeps dropping."
    },
]

In [0]:
batch_response = (
    workspace_client
    .serving_endpoints
    .query(
        name=SERVING_ENDPOINT_NAME,
        dataframe_records=(
            batch_tickets
        ),
    )
)

In [0]:
batch_prediction_df = (
    pd.DataFrame(
        batch_response.predictions
    )
)

display(
    batch_prediction_df
)

##### 10. Combine input and predictions

In [0]:
batch_input_df = (
    pd.DataFrame(
        batch_tickets
    )
)

batch_results_df = (
    pd.concat(
        [
            batch_input_df,
            batch_prediction_df,
        ],
        axis=1,
    )
)

display(
    batch_results_df
)

##### 11. Behavioral verification

In [0]:
expected_categories = [
    "Billing",
    "Cancellation",
    "Login",
    "Technical",
]

batch_results_df[
    "expected_category"
] = expected_categories

In [0]:
batch_results_df[
    "prediction_correct"
] = (
    batch_results_df[
        "predicted_category"
    ]
    ==
    batch_results_df[
        "expected_category"
    ]
)

display(
    batch_results_df[
        [
            "ticket_text",
            "expected_category",
            "predicted_category",
            "confidence",
            "prediction_correct",
        ]
    ]
)

In [0]:
num_correct = (
    batch_results_df[
        "prediction_correct"
    ]
    .sum()
)

num_tests = len(
    batch_results_df
)

print(
    f"Behavioral tests passed: "
    f"{num_correct}/{num_tests}"
)

In [0]:
assert (
    batch_results_df[
        "prediction_correct"
    ]
    .all()
), (
    "One or more endpoint behavioral tests failed."
)

print(
    "All endpoint behavioral  tests passed."
)

#These four cases are deployment sanity tests, not model evaluation.

##### 12. Verify batch probability sums

In [0]:
batch_probability_sums = (
    batch_prediction_df[
        probability_columns
    ]
    .sum(axis=1)
)

print(
    batch_probability_sums
)

In [0]:
assert (
    (
        batch_probability_sums
        - 1.0
    )
    .abs()
    .lt(1e-5)
    .all()
)

print(
    "All batch probability vectors sum to 1."
)

##### 13. Test an ambiguous ticket

In [0]:
ambiguous_ticket = {
    "ticket_text":
        "I cannot access my account and I am thinking about cancelling."
}

In [0]:
ambiguous_response = (
    workspace_client
    .serving_endpoints
    .query(
        name=SERVING_ENDPOINT_NAME,
        dataframe_records=[
            ambiguous_ticket
        ],
    )
)

In [0]:
ambiguous_prediction_df = (
    pd.DataFrame(
        ambiguous_response.predictions
    )
)

display(
    ambiguous_prediction_df
)

In [0]:
display(
    ambiguous_prediction_df[
        [
            "predicted_category",
            "confidence",
            "probability_Billing",
            "probability_Cancellation",
            "probability_Login",
            "probability_Technical",
        ]
    ]
)

##### 14. Final endpoint test summary

In [0]:
summary_df = pd.DataFrame(
    {
        "check": [
            "Endpoint ready",
            "Configuration complete",
            "Served entity available",
            "Single prediction",
            "Output schema",
            "Probability validation",
            "Batch prediction",
            "Behavioral tests",
            "Ambiguous-input test",
        ],
        "status": [
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            "PASS",
        ],
    }
)

display(
    summary_df
)

##### Key Learnings

1. Model registration and model serving are separate stages.

2. A registered model can be loaded successfully while an endpoint
   still fails because serving uses an isolated runtime environment.

3. Endpoint testing validates the complete production inference path,
   including packaging, dependencies, serving infrastructure, and
   model inference.

4. `dataframe_records` sends row-oriented tabular input to a custom
   MLflow model endpoint.

5. For this classifier, raw ticket text is sent directly to the
   serving endpoint. Tokenization and Transformer inference happen
   inside the packaged MLflow PyFunc.

6. Multiclass probabilities should sum to approximately 1.

7. The reported confidence is the maximum predicted class probability.
   It should not be interpreted as guaranteed certainty.

8. Batch testing confirms that the endpoint can process multiple
   tickets in one request.

9. Behavioral sanity tests complement formal model evaluation but do
   not replace held-out test-set metrics.

## Conclusion

The fine-tuned MiniLM support-ticket classifier has now completed the
full production lifecycle:

``` text

Raw support-ticket data
        ↓
NLP preprocessing
        ↓
TF-IDF baseline
        ↓
pretrained embeddings
        ↓
neural network classifier
        ↓
Transformer fine-tuning
        ↓
MLflow tracking
        ↓
Unity Catalog model registration
        ↓
Databricks Model Serving
        ↓
Endpoint inference testing

```
The deployed endpoint accepts raw support-ticket text and returns the
predicted support category, confidence, and class probabilities.

Successful endpoint testing confirms that the packaged Transformer
model can execute in the isolated Databricks Model Serving environment.

Four clear examples are deployment sanity tests, while formal model quality was already measured on the held-out test set.